# DTM Hyperparameter Tuning (tomotopy)

Grid-search hyperparameter tuning for tomotopy DTModel across subjects (cs, math, physics).
Evaluates **Coherence (C_v)**, **IRBO Diversity**, and **Topic Quality** (harmonic mean).
Best models are saved per subject by **Topic Quality**.

IRBO is computed from **aggregated word distributions** across all timepoints (holistic topic identity).
Per-timepoint min/max IRBO is also logged as diagnostic info.

## Imports & Configuration

In [1]:
import os
import gc
import ast
import time
import pandas as pd
import numpy as np
import tomotopy as tp
from pathlib import Path
from itertools import product, combinations
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"tomotopy version: {tp.isa}")

tomotopy version: avx2


In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../../../data/preprocess")
TUNNING_DIR = Path("../../../../results/dtm/tuning")
MODEL_DIR = Path("../../../../models/dtm/tuning")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (TUNNING_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Output directory: {TUNNING_DIR}")

Subjects: ['cs', 'math', 'physics']
Output directory: ../../../../results/dtm/tuning


## Hyperparameter Grid

In [3]:
PARAM_GRID = {
    "k": [50, 60 ,75],
    "alpha_var": [0.01, 0.1],
    "phi_var": [0.1],
    "train_iter": [500, 1000],
    "eta_var" : [0.01, 0.1],
}

FIXED_PARAMS = {
    "lr_a": 0.01,
    "lr_b": 0.1,
    "lr_c": 0.55,
    "min_cf": 5,
    "min_df": 3,
    "seed": 42,
}

ITERATION_STEP = 50

keys = list(PARAM_GRID.keys())
values = list(PARAM_GRID.values())
all_combos = list(product(*values))

print(f"Tunable parameters: {keys}")
print(f"Total parameter combinations: {len(all_combos)}")
print(f"Total runs (combinations x subjects): {len(all_combos) * len(LIST_SUBJECT)}")

Tunable parameters: ['k', 'alpha_var', 'phi_var', 'train_iter', 'eta_var']
Total parameter combinations: 24
Total runs (combinations x subjects): 72


## Helper Functions

In [4]:
def load_and_preprocess(subject: str):
    file_path = BASE_DIR / subject / "bow" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)

    if not pd.api.types.is_datetime64_any_dtype(df["submitted_date"]):
        df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    all_years = sorted(df["year"].unique())
    num_time_steps = len(all_years)
    year_to_timestep = {year: i for i, year in enumerate(all_years)}
    df["timestep"] = df["year"].map(year_to_timestep)

    token_list = df["text"].tolist()
    processed_docs = [ast.literal_eval(x) for x in token_list]
    doc_timesteps = df["timestep"].tolist()

    return df, processed_docs, doc_timesteps, num_time_steps, all_years


def build_and_train_dtm(processed_docs, doc_timesteps, num_time_steps, params):
    dt_model = tp.DTModel(
        t=num_time_steps,
        k=params["k"],
        alpha_var=params["alpha_var"],
        eta_var=params["eta_var"],
        phi_var=params["phi_var"],
        lr_a=FIXED_PARAMS["lr_a"],
        lr_b=FIXED_PARAMS["lr_b"],
        lr_c=FIXED_PARAMS["lr_c"],
        min_cf=FIXED_PARAMS["min_cf"],
        min_df=FIXED_PARAMS["min_df"],
        seed=FIXED_PARAMS["seed"],
    )

    for i in range(len(processed_docs)):
        if processed_docs[i]:
            dt_model.add_doc(processed_docs[i], timepoint=doc_timesteps[i])

    train_iter = params["train_iter"]
    for i in range(0, train_iter, ITERATION_STEP):
        dt_model.train(ITERATION_STEP)

    return dt_model


def calculate_coherence(model) -> float:
    """Calculate C_v coherence using tomotopy's built-in Coherence."""
    coherence = tp.coherence.Coherence(model, coherence="c_v", top_n=10)
    return coherence.get_score()


def get_aggregated_topic_words(model, top_n: int = 10):
    """
    Extract top-N words for each topic by averaging word probability
    distributions across ALL timepoints. This captures each topic's
    holistic identity over its entire lifetime.
    Uses model.used_vocabs (not model.vocabs) to match get_topic_word_dist output.
    """
    vocab_list = list(model.used_vocabs)
    num_tp = model.num_timepoints
    topics_words = []

    for topic_id in range(model.k):
        # Average word distribution across all timepoints
        avg_dist = np.zeros(len(vocab_list))
        for t in range(num_tp):
            avg_dist += np.array(model.get_topic_word_dist(topic_id, timepoint=t))
        avg_dist /= num_tp

        top_indices = np.argsort(avg_dist)[::-1][:top_n]
        words = [vocab_list[i] for i in top_indices]
        topics_words.append(words)

    return topics_words


def get_topic_words_at_timepoint(model, timepoint: int, top_n: int = 10):
    """Extract top-N words for each topic at a specific timepoint.
    Uses model.used_vocabs (not model.vocabs) to match get_topic_word_dist output.
    """
    vocab_list = list(model.used_vocabs)
    topics_words = []
    for topic_id in range(model.k):
        word_dist = model.get_topic_word_dist(topic_id, timepoint=timepoint)
        top_indices = np.argsort(word_dist)[::-1][:top_n]
        words = [vocab_list[i] for i in top_indices]
        topics_words.append(words)
    return topics_words


def rbo(list_1, list_2, p=0.9):
    """
    Rank-Biased Overlap (RBO) between two ranked lists.
    Returns similarity score in [0, 1]. Higher = more similar.
    """
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    
    rbo_score = 0.0
    for d in range(1, k + 1):
        set_1 = set(list_1[:d])
        set_2 = set(list_2[:d])
        agreement = len(set_1 & set_2) / d
        rbo_score += (p ** (d - 1)) * agreement
    
    rbo_score *= (1 - p)
    return rbo_score


def calculate_irbo(topics_words, p=0.9):
    """
    Calculate mean IRBO (Inverted RBO) diversity across all topic pairs.
    Returns mean_irbo in [0, 1]. Higher = more diverse.
    """
    if len(topics_words) < 2:
        return 0.0
    
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    
    return np.mean(irbo_scores)


def calculate_irbo_dtm(model, top_n: int = 10, p=0.9):
    """
    Calculate IRBO diversity for a DTModel.
    
    Returns:
        irbo_aggregated: IRBO from averaged word distributions (used for Topic Quality)
        irbo_tp_min: minimum per-timepoint IRBO (diagnostic)
        irbo_tp_max: maximum per-timepoint IRBO (diagnostic)
    """
    # Primary metric: IRBO from aggregated word distributions
    aggregated_words = get_aggregated_topic_words(model, top_n=top_n)
    irbo_aggregated = calculate_irbo(aggregated_words, p=p)

    # Diagnostic: per-timepoint IRBO min/max
    per_tp_irbo = []
    for t in range(model.num_timepoints):
        tp_words = get_topic_words_at_timepoint(model, timepoint=t, top_n=top_n)
        irbo = calculate_irbo(tp_words, p=p)
        per_tp_irbo.append(irbo)

    return irbo_aggregated, np.min(per_tp_irbo), np.max(per_tp_irbo)

## Load & Preprocess All Subjects

In [5]:
all_data = {}
all_processed_docs = {}
all_doc_timesteps = {}
all_num_time_steps = {}
all_years = {}

for subject in LIST_SUBJECT:
    print(f"Loading {subject}...")
    df, processed_docs, doc_timesteps, num_time_steps, years = load_and_preprocess(subject)
    all_data[subject] = df
    all_processed_docs[subject] = processed_docs
    all_doc_timesteps[subject] = doc_timesteps
    all_num_time_steps[subject] = num_time_steps
    all_years[subject] = years
    print(f"  {subject}: {len(df):,} documents, {num_time_steps} time steps ({years[0]}-{years[-1]})")

print(f"\nSubjects ready: {list(all_data.keys())}")

Loading cs...
  cs: 165,756 documents, 26 time steps (2000-2025)
Loading math...
  math: 157,085 documents, 26 time steps (2000-2025)
Loading physics...
  physics: 146,311 documents, 26 time steps (2000-2025)

Subjects ready: ['cs', 'math', 'physics']


## Hyperparameter Tuning Grid Search

For each parameter combination, compute **Coherence**, **IRBO Diversity** (from aggregated word distributions), and **Topic Quality** (harmonic mean).
Per-timepoint IRBO min/max is logged as diagnostic info.
Best models are saved per subject by Topic Quality.

In [6]:
total_combos = len(all_combos)

for subject in LIST_SUBJECT:
    processed_docs = all_processed_docs[subject]
    doc_timesteps = all_doc_timesteps[subject]
    num_time_steps = all_num_time_steps[subject]
    n_docs = len(all_data[subject])

    results_csv_path = TUNNING_DIR / subject / "tuning_results.csv"
    best_model_path = MODEL_DIR / subject / "best_model.bin"
    best_quality = -1.0

    # Resume support
    existing_results = []
    if results_csv_path.exists():
        existing_df = pd.read_csv(results_csv_path)
        existing_results = existing_df.to_dict("records")
        if len(existing_results) > 0:
            if "topic_quality" in existing_df.columns:
                best_quality = existing_df["topic_quality"].max()
            else:
                best_quality = existing_df["coherence_cv"].max()
            print(f"Resuming {subject}: {len(existing_results)} previous runs found, best quality so far: {best_quality:.4f}")

    results = existing_results.copy()

    completed_param_sets = set()
    for r in existing_results:
        param_key = (r["k"], r["alpha_var"], r["phi_var"], r["train_iter"], r.get("eta_var", 0.1))
        completed_param_sets.add(param_key)

    print(f"\n{'='*70}")
    print(f"Subject: {subject.upper()} ({n_docs:,} documents, {num_time_steps} time steps)")
    print(f"{'='*70}")

    for idx, combo in enumerate(all_combos, 1):
        params = dict(zip(keys, combo))

        param_key = (params["k"], params["alpha_var"], params["phi_var"], params["train_iter"], params["eta_var"])
        if param_key in completed_param_sets:
            continue

        print(f"\n[{idx}/{total_combos}] {subject} | "
              f"k={params['k']} \u03b1_var={params['alpha_var']} "
              f"\u03c6_var={params['phi_var']} iter={params['train_iter']} eta_var={params['eta_var']}")

        try:
            start_time = time.time()

            model = build_and_train_dtm(
                processed_docs, doc_timesteps, num_time_steps, params
            )
            coherence = calculate_coherence(model)

            # Compute IRBO: aggregated (for quality) + per-timepoint min/max (diagnostic)
            irbo_agg, irbo_tp_min, irbo_tp_max = calculate_irbo_dtm(model, top_n=TOP_N_WORDS, p=RBO_P)

            # Topic Quality = harmonic mean of coherence and aggregated IRBO
            if coherence + irbo_agg > 0:
                topic_quality = 2 * coherence * irbo_agg / (coherence + irbo_agg)
            else:
                topic_quality = 0.0

            elapsed = time.time() - start_time

            result_row = {
                "subject": subject,
                "k": params["k"],
                "alpha_var": params["alpha_var"],
                "phi_var": params["phi_var"],
                "train_iter": params["train_iter"],
                "eta_var": params["eta_var"],
                "lr_a": FIXED_PARAMS["lr_a"],
                "lr_b": FIXED_PARAMS["lr_b"],
                "lr_c": FIXED_PARAMS["lr_c"],
                "min_cf": FIXED_PARAMS["min_cf"],
                "min_df": FIXED_PARAMS["min_df"],
                "seed": FIXED_PARAMS["seed"],
                "coherence_cv": coherence,
                "irbo_aggregated": irbo_agg,
                "irbo_tp_min": irbo_tp_min,
                "irbo_tp_max": irbo_tp_max,
                "topic_quality": topic_quality,
                "ll_per_word": model.ll_per_word,
                "time_seconds": round(elapsed, 1),
            }
            results.append(result_row)

            is_new_best = topic_quality > best_quality
            if is_new_best:
                best_quality = topic_quality
                model.save(str(best_model_path))
                print(f"  \u2b50 NEW BEST | Quality: {topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_agg:.4f} [tp_min={irbo_tp_min:.4f}, tp_max={irbo_tp_max:.4f}]), eta_var={params['eta_var']} "
                      f"({elapsed:.1f}s) \u2192 Saved to {best_model_path}")
            else:
                print(f"  \u2713 Quality: {topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_agg:.4f} [tp_min={irbo_tp_min:.4f}, tp_max={irbo_tp_max:.4f}]) , eta_var={params['eta_var']}"
                      f"({elapsed:.1f}s) | Best: {best_quality:.4f}")

            # Save results incrementally
            pd.DataFrame(results).to_csv(results_csv_path, index=False)

            del model
            gc.collect()

        except Exception as e:
            print(f"  \u2717 ERROR: {e}")
            result_row = {
                "subject": subject,
                "k": params["k"],
                "alpha_var": params["alpha_var"],
                "phi_var": params["phi_var"],
                "train_iter": params["train_iter"],
                "eta_var": params["eta_var"],
                "lr_a": FIXED_PARAMS["lr_a"],
                "lr_b": FIXED_PARAMS["lr_b"],
                "lr_c": FIXED_PARAMS["lr_c"],
                "min_cf": FIXED_PARAMS["min_cf"],
                "min_df": FIXED_PARAMS["min_df"],
                "seed": FIXED_PARAMS["seed"],
                "coherence_cv": None,
                "irbo_aggregated": None,
                "irbo_tp_min": None,
                "irbo_tp_max": None,
                "topic_quality": None,
                "ll_per_word": None,
                "time_seconds": None,
            }
            results.append(result_row)
            pd.DataFrame(results).to_csv(results_csv_path, index=False)
            gc.collect()

    print(f"\n{'='*70}")
    print(f"\u2705 {subject.upper()} COMPLETE | Best quality: {best_quality:.4f}")
    print(f"Results saved to: {results_csv_path}")
    print(f"Best model saved to: {best_model_path}")
    print(f"{'='*70}")

Resuming cs: 28 previous runs found, best quality so far: 0.6100

Subject: CS (165,756 documents, 26 time steps)

✅ CS COMPLETE | Best quality: 0.6100
Results saved to: ../../../../results/dtm/tuning/cs/tuning_results.csv
Best model saved to: ../../../../models/dtm/tuning/cs/best_model.bin
Resuming math: 8 previous runs found, best quality so far: 0.5905

Subject: MATH (157,085 documents, 26 time steps)

[1/24] math | k=50 α_var=0.01 φ_var=0.1 iter=500 eta_var=0.01


/tmp/ipykernel_301303/1205574847.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  dt_model.train(ITERATION_STEP)


  ✓ Quality: 0.5877 (C=0.4291, IRBO=0.9325 [tp_min=0.6620, tp_max=0.9824]) , eta_var=0.01(459.1s) | Best: 0.5905

[3/24] math | k=50 α_var=0.01 φ_var=0.1 iter=1000 eta_var=0.01
  ⭐ NEW BEST | Quality: 0.5942 (C=0.4309, IRBO=0.9572 [tp_min=0.7025, tp_max=0.9893]), eta_var=0.01 (882.7s) → Saved to ../../../../models/dtm/tuning/math/best_model.bin

[4/24] math | k=50 α_var=0.01 φ_var=0.1 iter=1000 eta_var=0.1
  ✓ Quality: 0.5902 (C=0.4269, IRBO=0.9559 [tp_min=0.7086, tp_max=0.9899]) , eta_var=0.1(888.6s) | Best: 0.5942

[5/24] math | k=50 α_var=0.1 φ_var=0.1 iter=500 eta_var=0.01
  ✓ Quality: 0.5861 (C=0.4282, IRBO=0.9289 [tp_min=0.6686, tp_max=0.9832]) , eta_var=0.01(451.4s) | Best: 0.5942

[7/24] math | k=50 α_var=0.1 φ_var=0.1 iter=1000 eta_var=0.01
  ✓ Quality: 0.5882 (C=0.4248, IRBO=0.9558 [tp_min=0.6993, tp_max=0.9901]) , eta_var=0.01(866.6s) | Best: 0.5942

[8/24] math | k=50 α_var=0.1 φ_var=0.1 iter=1000 eta_var=0.1
  ⭐ NEW BEST | Quality: 0.5943 (C=0.4308, IRBO=0.9573 [tp_min=0.7

/tmp/ipykernel_301303/1205574847.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  dt_model.train(ITERATION_STEP)


  ⭐ NEW BEST | Quality: 0.5748 (C=0.4119, IRBO=0.9508 [tp_min=0.7057, tp_max=0.9886]), eta_var=0.01 (587.9s) → Saved to ../../../../models/dtm/tuning/physics/best_model.bin

[3/24] physics | k=50 α_var=0.01 φ_var=0.1 iter=1000 eta_var=0.01
  ⭐ NEW BEST | Quality: 0.5749 (C=0.4095, IRBO=0.9646 [tp_min=0.7124, tp_max=0.9934]), eta_var=0.01 (1092.9s) → Saved to ../../../../models/dtm/tuning/physics/best_model.bin

[4/24] physics | k=50 α_var=0.01 φ_var=0.1 iter=1000 eta_var=0.1
  ⭐ NEW BEST | Quality: 0.5781 (C=0.4125, IRBO=0.9662 [tp_min=0.7038, tp_max=0.9935]), eta_var=0.1 (1110.0s) → Saved to ../../../../models/dtm/tuning/physics/best_model.bin

[5/24] physics | k=50 α_var=0.1 φ_var=0.1 iter=500 eta_var=0.01
  ⭐ NEW BEST | Quality: 0.5788 (C=0.4159, IRBO=0.9515 [tp_min=0.6993, tp_max=0.9883]), eta_var=0.01 (587.7s) → Saved to ../../../../models/dtm/tuning/physics/best_model.bin

[7/24] physics | k=50 α_var=0.1 φ_var=0.1 iter=1000 eta_var=0.01
  ✓ Quality: 0.5767 (C=0.4111, IRBO=0.9658 

## Summary: Best Parameters Per Subject (by Topic Quality)

In [7]:
print("\n" + "=" * 110)
print("FINAL RESULTS: Best Parameters Per Subject (by Topic Quality)")
print("=" * 110)

for subject in LIST_SUBJECT:
    results_csv_path = TUNNING_DIR / subject / "tuning_results.csv"
    if not results_csv_path.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    df = pd.read_csv(results_csv_path)
    df_valid = df.dropna(subset=["coherence_cv"])

    if len(df_valid) == 0:
        print(f"\n{subject.upper()}: No valid results")
        continue

    best_row = df_valid.loc[df_valid["topic_quality"].idxmax()]

    sep = "\u2500" * 50
    print(f"  Subject:          {subject.upper()}")
    print(f"  Total runs:       {len(df_valid)}")
    print(f"  Best Quality:     {best_row['topic_quality']:.4f}")
    print(f"  Coherence:        {best_row['coherence_cv']:.4f}")
    print(f"  IRBO (aggregated):{best_row['irbo_aggregated']:.4f}")
    print(f"  IRBO tp range:    [{best_row['irbo_tp_min']:.4f}, {best_row['irbo_tp_max']:.4f}]")
    print(f"  Parameters:")
    print(f"    k          = {int(best_row['k'])}")
    print(f"    alpha_var  = {best_row['alpha_var']}")
    print(f"    phi_var    = {best_row['phi_var']}")
    print(f"    train_iter = {int(best_row['train_iter'])}")
    print(f"    min_cf     = {int(best_row['min_cf'])}")
    print(f"    min_df     = {int(best_row['min_df'])}")
    sep = "\u2500" * 50

print("\n" + "=" * 110)
print("Top 5 configurations per subject (by Topic Quality):")
print("=" * 110)

for subject in LIST_SUBJECT:
    results_csv_path = TUNNING_DIR / subject / "tuning_results.csv"
    if not results_csv_path.exists():
        continue

    df = pd.read_csv(results_csv_path)
    df_valid = df.dropna(subset=["coherence_cv"]).sort_values("topic_quality", ascending=False)

    print(f"\n{subject.upper()}:")
    print(df_valid[["k", "alpha_var", "phi_var", "train_iter", "coherence_cv",
                    "irbo_aggregated", "irbo_tp_min", "irbo_tp_max",
                    "topic_quality", "time_seconds"]].head(5).to_string(index=False))
    print()


FINAL RESULTS: Best Parameters Per Subject (by Topic Quality)
  Subject:          CS
  Total runs:       28
  Best Quality:     0.6100
  Coherence:        0.4426
  IRBO (aggregated):0.9811
  IRBO tp range:    [0.6718, 0.9988]
  Parameters:
    k          = 50
    alpha_var  = 0.1
    phi_var    = 0.1
    train_iter = 1000
    min_cf     = 5
    min_df     = 3
  Subject:          MATH
  Total runs:       28
  Best Quality:     0.5943
  Coherence:        0.4308
  IRBO (aggregated):0.9573
  IRBO tp range:    [0.7189, 0.9913]
  Parameters:
    k          = 50
    alpha_var  = 0.1
    phi_var    = 0.1
    train_iter = 1000
    min_cf     = 5
    min_df     = 3
  Subject:          PHYSICS
  Total runs:       28
  Best Quality:     0.5800
  Coherence:        0.4157
  IRBO (aggregated):0.9591
  IRBO tp range:    [0.7108, 0.9891]
  Parameters:
    k          = 60
    alpha_var  = 0.01
    phi_var    = 0.1
    train_iter = 1000
    min_cf     = 5
    min_df     = 3

Top 5 configurations per sub